In [ ]:
# CELL 01 — Load and preview the raw CSV; leave the source data unchanged.

from pathlib import Path
import pandas as pd

# Find data/raw/BankChurners.csv from this notebook or a parent folder
data_path = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidate = folder / "data" / "raw" / "BankChurners.csv"
    if candidate.is_file():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/BankChurners.csv. "
        "Check the file name and project folder."
    )

df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Shape:", df.shape)
display(df.head())

Loaded: /Users/Claracodess/Documents/MIVA  /COURSES/MIT/THIRD_SEMESTER/MIS/MIT 8339 - Business Intelligence and Decision Support Systems/MIT8339_BI_LAB_CA/data/raw/BankChurners.csv
Shape: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [ ]:
# CELL 02 — Profile structure, missing values, duplicates, IDs, attrition balance, and Unknown categories.

print("Data types and non-null counts:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing"))

print("Duplicate rows:", df.duplicated().sum())
print("Unique customer IDs:", df["CLIENTNUM"].nunique())

print("\nAttrition counts:")
display(df["Attrition_Flag"].value_counts().to_frame("customers"))

print("\nAttrition percentages:")
display(
    (df["Attrition_Flag"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percent")
)

print("\nExplicit 'Unknown' values:")
for column in ["Education_Level", "Marital_Status", "Income_Category"]:
    if column in df.columns:
        print(f"{column}: {(df[column] == 'Unknown').sum()}")

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  str    
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                      

,missing
CLIENTNUM,0
Attrition_Flag,0
Customer_Age,0
Gender,0
Dependent_count,0
Education_Level,0
Marital_Status,0
Income_Category,0
Card_Category,0
Months_on_book,0


Duplicate rows: 0
Unique customer IDs: 10127

Attrition counts:


,customers
Attrition_Flag,
Existing Customer,8500
Attrited Customer,1627



Attrition percentages:


,percent
Attrition_Flag,
Existing Customer,83.93
Attrited Customer,16.07



Explicit 'Unknown' values:
Education_Level: 1519
Marital_Status: 749
Income_Category: 1112


In [ ]:
# CELL 03 — Validate labels, numeric ranges, IQR flags, and domain consistency.

categorical_cols = [
    "Attrition_Flag", "Gender", "Education_Level",
    "Marital_Status", "Income_Category", "Card_Category"
]

for column in categorical_cols:
    values = sorted(df[column].dropna().unique())
    text_values = df[column].astype("string")
    whitespace_rows = text_values.str.strip().ne(text_values).sum()

    print(f"\n{column}: {values}")
    print("Rows with leading/trailing spaces:", int(whitespace_rows))


# Summarise numeric fields, excluding the ID and precomputed classifier columns
leakage_cols = [
    col for col in df.columns
    if col.startswith("Naive_Bayes_Classifier_")
]

numeric_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col != "CLIENTNUM" and col not in leakage_cols
]

display(
    df[numeric_cols]
    .describe()
    .T[["min", "25%", "50%", "75%", "max"]]
    .round(3)
)


# Flag possible IQR outliers for investigation—not automatic removal
iqr_flags = {}

for column in numeric_cols:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    iqr_flags[column] = int(((df[column] < low) | (df[column] > high)).sum())

display(
    pd.Series(iqr_flags, name="IQR_screening_flags")
    .sort_values(ascending=False)
    .to_frame()
)


# Check a basic arithmetic relationship between credit-limit fields
reconciliation_gap = (
    df["Credit_Limit"]
    - df["Total_Revolving_Bal"]
    - df["Avg_Open_To_Buy"]
).abs()

print(
    "Rows where Credit_Limit does not reconcile "
    "(tolerance 0.01):",
    int((reconciliation_gap > 0.01).sum())
)
print("Maximum reconciliation gap:", round(reconciliation_gap.max(), 4))

print(
    "Utilisation values outside [0, 1]:",
    int((~df["Avg_Utilization_Ratio"].between(0, 1)).sum())
)


Attrition_Flag: ['Attrited Customer', 'Existing Customer']
Rows with leading/trailing spaces: 0

Gender: ['F', 'M']
Rows with leading/trailing spaces: 0

Education_Level: ['College', 'Doctorate', 'Graduate', 'High School', 'Post-Graduate', 'Uneducated', 'Unknown']
Rows with leading/trailing spaces: 0

Marital_Status: ['Divorced', 'Married', 'Single', 'Unknown']
Rows with leading/trailing spaces: 0

Income_Category: ['$120K +', '$40K - $60K', '$60K - $80K', '$80K - $120K', 'Less than $40K', 'Unknown']
Rows with leading/trailing spaces: 0

Card_Category: ['Blue', 'Gold', 'Platinum', 'Silver']
Rows with leading/trailing spaces: 0


,min,25%,50%,75%,max
Customer_Age,26.0,41.000,46.000,52.000,73.000
Dependent_count,0.0,1.000,2.000,3.000,5.000
Months_on_book,13.0,31.000,36.000,40.000,56.000
Total_Relationship_Count,1.0,3.000,4.000,5.000,6.000
Months_Inactive_12_mon,0.0,2.000,2.000,3.000,6.000
Contacts_Count_12_mon,0.0,2.000,2.000,3.000,6.000
Credit_Limit,1438.3,2555.000,4549.000,11067.500,34516.000
Total_Revolving_Bal,0.0,359.000,1276.000,1784.000,2517.000
Avg_Open_To_Buy,3.0,1324.500,3474.000,9859.000,34516.000
Total_Amt_Chng_Q4_Q1,0.0,0.631,0.736,0.859,3.397


,IQR_screening_flags
Credit_Limit,984
Avg_Open_To_Buy,963
Total_Trans_Amt,896
Contacts_Count_12_mon,629
Total_Amt_Chng_Q4_Q1,396
Total_Ct_Chng_Q4_Q1,394
Months_on_book,386
Months_Inactive_12_mon,331
Customer_Age,2
Total_Trans_Ct,2


Rows where Credit_Limit does not reconcile (tolerance 0.01): 0
Maximum reconciliation gap: 0.0
Utilisation values outside [0, 1]: 0


In [ ]:
# CELL 04 — Create and verify age bands on a working copy.

df_work = df.copy()

df_work["Age_Band"] = pd.cut(
    df_work["Customer_Age"],
    bins=[-float("inf"), 34, 44, 54, float("inf")],
    labels=["Under 35", "35–44", "45–54", "55+"],
    include_lowest=True
)

display(
    df_work["Age_Band"]
    .value_counts(sort=False, dropna=False)
    .to_frame("customers")
)

print("Rows without an age band:", int(df_work["Age_Band"].isna().sum()))

,customers
Age_Band,
Under 35,735
35–44,3440
45–54,4342
55+,1610


Rows without an age band: 0


In [ ]:
# CELL 05 — Create and verify sample-relative utilisation bands.

util_q1, util_q3 = df_work["Avg_Utilization_Ratio"].quantile([0.25, 0.75])

df_work["Utilization_Band"] = pd.cut(
    df_work["Avg_Utilization_Ratio"],
    bins=[-float("inf"), util_q1, util_q3, float("inf")],
    labels=["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"],
    include_lowest=True
)

print("Q1:", round(util_q1, 3), "| Q3:", round(util_q3, 3))

display(
    df_work["Utilization_Band"]
    .value_counts(sort=False, dropna=False)
    .to_frame("customers")
)

print(
    "Rows without a utilisation band:",
    int(df_work["Utilization_Band"].isna().sum())
)

Q1: 0.023 | Q3: 0.503


,customers
Utilization_Band,
Low (≤Q1),2541
Middle (Q1–Q3),5058
High (>Q3),2528


Rows without a utilisation band: 0


In [ ]:
# CELL 06 — Create and verify sample-relative transaction-frequency bands.

trans_q1, trans_q3 = df_work["Total_Trans_Ct"].quantile([0.25, 0.75])

df_work["Transaction_Frequency_Band"] = pd.cut(
    df_work["Total_Trans_Ct"],
    bins=[-float("inf"), trans_q1, trans_q3, float("inf")],
    labels=["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"],
    include_lowest=True
)

print("Q1:", trans_q1, "| Q3:", trans_q3)

display(
    df_work["Transaction_Frequency_Band"]
    .value_counts(sort=False, dropna=False)
    .to_frame("customers")
)

print(
    "Rows without a transaction-frequency band:",
    int(df_work["Transaction_Frequency_Band"].isna().sum())
)

Q1: 45.0 | Q3: 81.0


,customers
Transaction_Frequency_Band,
Low (≤Q1),2611
Middle (Q1–Q3),5174
High (>Q3),2342


Rows without a transaction-frequency band: 0


In [ ]:
# CELL 07 — Build and validate the analytical table; exclude the precomputed classifier columns.

leakage_cols = [
    col for col in df_work.columns
    if col.startswith("Naive_Bayes_Classifier_")
]

df_clean = df_work.drop(columns=leakage_cols).copy()

print("Columns excluded:", leakage_cols)
print("Raw shape:", df.shape)
print("Analysis-table shape:", df_clean.shape)
print("Rows removed:", len(df_work) - len(df_clean))
print("Nulls in analysis table:", int(df_clean.isna().sum().sum()))
print("Duplicate rows:", int(df_clean.duplicated().sum()))
print("Unique customer IDs:", df_clean["CLIENTNUM"].nunique())

print("\nAttrition counts in analysis table:")
display(df_clean["Attrition_Flag"].value_counts().to_frame("customers"))

print("\nMissing values in the three derived fields:")
display(
    df_clean[
        ["Age_Band", "Utilization_Band", "Transaction_Frequency_Band"]
    ].isna().sum().to_frame("missing")
)

Columns excluded: ['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']
Raw shape: (10127, 23)
Analysis-table shape: (10127, 24)
Rows removed: 0
Nulls in analysis table: 0
Duplicate rows: 0
Unique customer IDs: 10127

Attrition counts in analysis table:


,customers
Attrition_Flag,
Existing Customer,8500
Attrited Customer,1627



Missing values in the three derived fields:


,missing
Age_Band,0
Utilization_Band,0
Transaction_Frequency_Band,0


In [ ]:
# CELL 08 — Export the cleaned CSV and verify it by reloading.

output_path = data_path.parent.parent / "processed" / "BankChurners_clean.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

# Read it back to verify the export
df_exported = pd.read_csv(output_path)

print("Saved to:", output_path)
print("Reloaded shape:", df_exported.shape)
print("Nulls after reload:", int(df_exported.isna().sum().sum()))
print("Duplicate rows after reload:", int(df_exported.duplicated().sum()))
print("Unique customer IDs:", df_exported["CLIENTNUM"].nunique())

Saved to: /Users/Claracodess/Documents/MIVA  /COURSES/MIT/THIRD_SEMESTER/MIS/MIT 8339 - Business Intelligence and Decision Support Systems/MIT8339_BI_LAB_CA/data/processed/BankChurners_clean.csv
Reloaded shape: (10127, 24)
Nulls after reload: 0
Duplicate rows after reload: 0
Unique customer IDs: 10127


In [ ]:
# CELL 09 — Calculate whole-dataset KPI baselines.

clean_path = data_path.parent.parent / "processed" / "BankChurners_clean.csv"
df_clean = pd.read_csv(clean_path)

attrited = df_clean["Attrition_Flag"].eq("Attrited Customer")

portfolio_kpis = pd.Series({
    "Unique customers": df_clean["CLIENTNUM"].nunique(),
    "Attrited customers": int(attrited.sum()),
    "Attrition rate (%)": attrited.mean() * 100,
    "Total transaction amount (dataset units)": df_clean["Total_Trans_Amt"].sum(),
    "Mean transaction amount/customer (dataset units)": df_clean["Total_Trans_Amt"].mean(),
    "Mean transaction count/customer": df_clean["Total_Trans_Ct"].mean(),
    "Mean customer-level utilisation (%)": df_clean["Avg_Utilization_Ratio"].mean() * 100,
    "Mean product relationship count": df_clean["Total_Relationship_Count"].mean(),
    "Mean months inactive": df_clean["Months_Inactive_12_mon"].mean(),
    "Mean credit limit (dataset units)": df_clean["Credit_Limit"].mean()
}, name="value")

display(portfolio_kpis.round(2).to_frame())

,value
Unique customers,10127.00
Attrited customers,1627.00
Attrition rate (%),16.07
Total transaction amount (dataset units),44600182.00
Mean transaction amount/customer (dataset units),4404.09
Mean transaction count/customer,64.86
Mean customer-level utilisation (%),27.49
Mean product relationship count,3.81
Mean months inactive,2.34
Mean credit limit (dataset units),8631.95


In [12]:

# CELL 10 — Compare attrited and existing customers.
# Show group sizes, means, and medians for engagement, product, transaction,
# and utilisation variables. This is descriptive, not causal.

comparison_vars = [
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Relationship_Count",
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Total_Ct_Chng_Q4_Q1",
    "Avg_Utilization_Ratio"
]

# Report group sizes so comparisons are interpreted with their denominators.
display(
    df_clean["Attrition_Flag"]
    .value_counts()
    .rename("customers")
    .to_frame()
)

# Compare means and medians for each selected variable by attrition status.
summary_by_attrition = (
    df_clean.groupby("Attrition_Flag")[comparison_vars]
    .agg(["mean", "median"])
    .T
)

display(summary_by_attrition.round(2))

# Show the mean difference, calculated as attrited minus existing.
mean_by_group = df_clean.groupby("Attrition_Flag")[comparison_vars].mean()

mean_difference = (
    mean_by_group.loc["Attrited Customer"]
    - mean_by_group.loc["Existing Customer"]
).rename("Attrited minus Existing").to_frame()

display(mean_difference.round(2))

,customers
Attrition_Flag,
Existing Customer,8500
Attrited Customer,1627


Attrition_Flag                   Attrited Customer  Existing Customer
Months_Inactive_12_mon   mean                 2.69               2.27
                         median               3.00               2.00
Contacts_Count_12_mon    mean                 2.97               2.36
                         median               3.00               2.00
Total_Relationship_Count mean                 3.28               3.91
                         median               3.00               4.00
Total_Trans_Ct           mean                44.93              68.67
                         median              43.00              71.00
Total_Trans_Amt          mean              3095.03            4654.66
                         median            2329.00            4100.00
Total_Ct_Chng_Q4_Q1      mean                 0.55               0.74
                         median               0.53               0.72
Avg_Utilization_Ratio    mean                 0.16               0.30
                         median               0.00               0.21

,Attrited minus Existing
Months_Inactive_12_mon,0.42
Contacts_Count_12_mon,0.62
Total_Relationship_Count,-0.63
Total_Trans_Ct,-23.74
Total_Trans_Amt,-1559.63
Total_Ct_Chng_Q4_Q1,-0.19
Avg_Utilization_Ratio,-0.13


In [13]:
# CELL 11 — Calculate attrition rates within each age band.
# The denominator is the number of unique customers in that band.
# These are observed sample rates, not predictions of future churn.

age_attrition = (
    df_clean.assign(
        _Is_Attrited=df_clean["Attrition_Flag"].eq("Attrited Customer")
    )
    .groupby("Age_Band", dropna=False)
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=("_Is_Attrited", "sum")
    )
)

age_attrition["attrition_rate_pct"] = (
    age_attrition["attrited"] / age_attrition["customers"] * 100
)

# Keep the age bands in the intended order for interpretation and charting.
age_order = ["Under 35", "35–44", "45–54", "55+"]
age_attrition = age_attrition.reindex(age_order)

display(age_attrition.round(2))

,customers,attrited,attrition_rate_pct
Age_Band,,,
Under 35,735,101,13.74
35–44,3440,548,15.93
45–54,4342,716,16.49
55+,1610,262,16.27


In [14]:
# CELL 12 — Calculate observed attrition within each utilisation band.
# Denominator: unique customers in that band.
# These sample-relative rates are not future churn probabilities or policy thresholds.

utilisation_attrition = (
    df_clean.assign(
        _Is_Attrited=df_clean["Attrition_Flag"].eq("Attrited Customer")
    )
    .groupby("Utilization_Band", dropna=False)
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=("_Is_Attrited", "sum")
    )
)

utilisation_attrition["attrition_rate_pct"] = (
    utilisation_attrition["attrited"]
    / utilisation_attrition["customers"]
    * 100
)

# Keep the bands ordered from low to high utilisation.
utilisation_order = ["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"]
utilisation_attrition = utilisation_attrition.reindex(utilisation_order)

display(utilisation_attrition.round(2))

,customers,attrited,attrition_rate_pct
Utilization_Band,,,
Low (≤Q1),2541,930,36.60
Middle (Q1–Q3),5058,472,9.33
High (>Q3),2528,225,8.90


In [15]:
# CELL 13 — Calculate observed attrition within each transaction-frequency band.
# Denominator: unique customers in that band.
# The bands are sample-relative; these rates are not future churn probabilities.

frequency_attrition = (
    df_clean.assign(
        _Is_Attrited=df_clean["Attrition_Flag"].eq("Attrited Customer")
    )
    .groupby("Transaction_Frequency_Band", dropna=False)
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=("_Is_Attrited", "sum")
    )
)

frequency_attrition["attrition_rate_pct"] = (
    frequency_attrition["attrited"]
    / frequency_attrition["customers"]
    * 100
)

# Keep the bands in their intended low-to-high order.
frequency_order = ["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"]
frequency_attrition = frequency_attrition.reindex(frequency_order)

display(frequency_attrition.round(2))

,customers,attrited,attrition_rate_pct
Transaction_Frequency_Band,,,
Low (≤Q1),2611,968,37.07
Middle (Q1–Q3),5174,633,12.23
High (>Q3),2342,26,1.11


In [16]:
# CELL 14 — Calculate observed attrition within each income category.
# Denominator: unique customers in the category.
# Income labels remain as supplied by the dataset.

income_attrition = (
    df_clean.assign(
        _Is_Attrited=df_clean["Attrition_Flag"].eq("Attrited Customer")
    )
    .groupby("Income_Category", dropna=False)
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=("_Is_Attrited", "sum")
    )
)

income_attrition["attrition_rate_pct"] = (
    income_attrition["attrited"]
    / income_attrition["customers"]
    * 100
)

# Display income categories in their source order, including Unknown.
income_order = [
    "Less than $40K",
    "$40K - $60K",
    "$60K - $80K",
    "$80K - $120K",
    "$120K +",
    "Unknown"
]
income_attrition = income_attrition.reindex(income_order)

display(income_attrition.round(2))

,customers,attrited,attrition_rate_pct
Income_Category,,,
Less than $40K,3561,612,17.19
$40K - $60K,1790,271,15.14
$60K - $80K,1402,189,13.48
$80K - $120K,1535,242,15.77
$120K +,727,126,17.33
Unknown,1112,187,16.82


In [17]:
# CELL 15 — Calculate observed attrition within each card category.
# Show each category's denominator because rare card types can have unstable rates.

card_attrition = (
    df_clean.assign(
        _Is_Attrited=df_clean["Attrition_Flag"].eq("Attrited Customer")
    )
    .groupby("Card_Category", dropna=False)
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=("_Is_Attrited", "sum")
    )
)

card_attrition["attrition_rate_pct"] = (
    card_attrition["attrited"]
    / card_attrition["customers"]
    * 100
)

# Keep the categories in a readable product order.
card_order = ["Blue", "Silver", "Gold", "Platinum"]
card_attrition = card_attrition.reindex(card_order)

display(card_attrition.round(2))

,customers,attrited,attrition_rate_pct
Card_Category,,,
Blue,9436,1519,16.10
Silver,555,82,14.77
Gold,116,21,18.10
Platinum,20,5,25.00


In [18]:
# CELL 16 — Examine utilisation and transaction frequency jointly.
# Show customer counts and observed attrition within each combined segment.
# This checks overlap; it does not estimate a causal or future-churn effect.

utilisation_order = ["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"]
frequency_order = ["Low (≤Q1)", "Middle (Q1–Q3)", "High (>Q3)"]

# Mark the target as 1 for attrited customers and 0 for existing customers.
is_attrited = df_clean["Attrition_Flag"].eq("Attrited Customer").astype(int)

# Count all customers in each utilisation × frequency combination.
joint_customers = pd.crosstab(
    df_clean["Utilization_Band"],
    df_clean["Transaction_Frequency_Band"]
).reindex(
    index=utilisation_order,
    columns=frequency_order,
    fill_value=0
)

# Count attrited customers in the same combinations.
joint_attrited = pd.crosstab(
    df_clean["Utilization_Band"],
    df_clean["Transaction_Frequency_Band"],
    values=is_attrited,
    aggfunc="sum"
).reindex(
    index=utilisation_order,
    columns=frequency_order,
    fill_value=0
)

# Calculate the attrition rate using each combined segment's customer count.
joint_rates_pct = (
    joint_attrited
    .div(joint_customers.where(joint_customers.ne(0)))
    * 100
)

print("Customers in each combined segment:")
display(joint_customers)

print("Attrited customers in each combined segment:")
display(joint_attrited)

print("Observed attrition rate within each combined segment (%):")
display(joint_rates_pct.round(2))

Customers in each combined segment:


Transaction_Frequency_Band,Low (≤Q1),Middle (Q1–Q3),High (>Q3)
Utilization_Band,,,
Low (≤Q1),764,1331,446
Middle (Q1–Q3),1253,2444,1361
High (>Q3),594,1399,535


Attrited customers in each combined segment:


Transaction_Frequency_Band,Low (≤Q1),Middle (Q1–Q3),High (>Q3)
Utilization_Band,,,
Low (≤Q1),559,357,14
Middle (Q1–Q3),267,194,11
High (>Q3),142,82,1


Observed attrition rate within each combined segment (%):


Transaction_Frequency_Band,Low (≤Q1),Middle (Q1–Q3),High (>Q3)
Utilization_Band,,,
Low (≤Q1),73.17,26.82,3.14
Middle (Q1–Q3),21.31,7.94,0.81
High (>Q3),23.91,5.86,0.19


In [19]:
# CELL 17 — Create and evaluate the rule-based retention-priority bands.
# One point for low transaction frequency and one for low utilisation.
# These bands summarise observed sample patterns; they are not churn probabilities.

df_priority = df_clean.copy()

# Create transparent 0/1 flags using the sample-relative cut-offs.
df_priority["Low_Transaction_Flag"] = (
    df_priority["Transaction_Frequency_Band"] == "Low (≤Q1)"
).astype(int)

df_priority["Low_Utilization_Flag"] = (
    df_priority["Utilization_Band"] == "Low (≤Q1)"
).astype(int)

# Score of 0, 1, or 2 equals the number of low-activity flags present.
df_priority["Retention_Priority_Score"] = (
    df_priority["Low_Transaction_Flag"]
    + df_priority["Low_Utilization_Flag"]
)

# Map the score to a descriptive priority tier.
priority_labels = {0: "Low", 1: "Medium", 2: "High"}
df_priority["Retention_Priority_Band"] = (
    df_priority["Retention_Priority_Score"].map(priority_labels)
)

# Evaluate the observed attrition rate within each priority tier.
priority_summary = (
    df_priority.groupby("Retention_Priority_Band")
    .agg(
        customers=("CLIENTNUM", "nunique"),
        attrited=(
            "Attrition_Flag",
            lambda values: values.eq("Attrited Customer").sum()
        )
    )
)

priority_summary["attrition_rate_pct"] = (
    priority_summary["attrited"]
    / priority_summary["customers"]
    * 100
)

# Display high priority first, then medium and low.
priority_summary = priority_summary.reindex(["High", "Medium", "Low"])

display(priority_summary.round(2))
print(
    "Rows without a priority band:",
    int(df_priority["Retention_Priority_Band"].isna().sum())
)

,customers,attrited,attrition_rate_pct
Retention_Priority_Band,,,
High,764,559,73.17
Medium,3624,780,21.52
Low,5739,288,5.02


Rows without a priority band: 0


In [20]:
# CELL 18 — Export and verify the final analysis-ready file.
# Preserve BankChurners_clean.csv; save a separate copy that includes the
# retention-priority flags, score, and band.

analysis_path = (
    data_path.parent.parent
    / "processed"
    / "BankChurners_analysis_ready.csv"
)
analysis_path.parent.mkdir(parents=True, exist_ok=True)

# Save the table with the derived fields and retention-priority classification.
df_priority.to_csv(analysis_path, index=False)

# Reload the export and verify its structure and key quality checks.
df_final = pd.read_csv(analysis_path)

print("Saved file:", analysis_path.name)
print("Shape:", df_final.shape)
print("Nulls:", int(df_final.isna().sum().sum()))
print("Duplicate rows:", int(df_final.duplicated().sum()))
print("Unique customer IDs:", df_final["CLIENTNUM"].nunique())

print("\nRetention-priority counts after reload:")
display(
    df_final["Retention_Priority_Band"]
    .value_counts()
    .reindex(["High", "Medium", "Low"])
    .to_frame("customers")
)

Saved file: BankChurners_analysis_ready.csv
Shape: (10127, 28)
Nulls: 0
Duplicate rows: 0
Unique customer IDs: 10127

Retention-priority counts after reload:


,customers
Retention_Priority_Band,
High,764
Medium,3624
Low,5739


In [23]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

print("Shape:", df_priority.shape)
print()
print(df_priority.dtypes.to_string())
print()
print("Columns 1-14")
print(df_priority.head(10).iloc[:, :14].to_string())
print()
print("Columns 15-28")
print(df_priority.head(10).iloc[:, 14:].to_string())

Shape: (10127, 28)

CLIENTNUM                       int64
Attrition_Flag                    str
Customer_Age                    int64
Gender                            str
Dependent_count                 int64
Education_Level                   str
Marital_Status                    str
Income_Category                   str
Card_Category                     str
Months_on_book                  int64
Total_Relationship_Count        int64
Months_Inactive_12_mon          int64
Contacts_Count_12_mon           int64
Credit_Limit                  float64
Total_Revolving_Bal             int64
Avg_Open_To_Buy               float64
Total_Amt_Chng_Q4_Q1          float64
Total_Trans_Amt                 int64
Total_Trans_Ct                  int64
Total_Ct_Chng_Q4_Q1           float64
Avg_Utilization_Ratio         float64
Age_Band                          str
Utilization_Band                  str
Transaction_Frequency_Band        str
Low_Transaction_Flag            int64
Low_Utilization_Flag          

In [24]:
import platform, importlib.metadata as md

print("| Component | Version |")
print("|---|---|")
print(f"| Python | {platform.python_version()} |")
for pkg in ["pandas", "numpy", "plotly", "dash", "notebook", "jupyterlab", "ipykernel"]:
    try:
        print(f"| {pkg} | {md.version(pkg)} |")
    except md.PackageNotFoundError:
        pass
print(f"| Operating system | {platform.system()} {platform.release()} |")

| Component | Version |
|---|---|
| Python | 3.13.1 |
| pandas | 3.0.6 |
| numpy | 2.5.4 |
| plotly | 7.1.0 |
| dash | 4.4.1 |
| notebook | 7.6.3 |
| jupyterlab | 4.6.4 |
| ipykernel | 7.4.0 |
| Operating system | Darwin 25.5.0 |
